STEP 1: duplicate row detection

In [1]:
import pandas as pd

df = pd.read_csv(
    "All_Tourism_Reviews.csv")



In [2]:
duplicates = df.duplicated()

print("Total rows:", len(df))
print("Exact duplicates:", duplicates.sum())

Total rows: 22344
Exact duplicates: 1


In [3]:
duplicate_rows = df[df.duplicated(keep=False)]

print(duplicate_rows)

        City         Place  Page                             Review_Text  \
16081  Paris  Eiffel Tower    33  You've seen one, you've seen them all.   
16082  Paris  Eiffel Tower    33  You've seen one, you've seen them all.   

              Review_Date Reviewer_Location  Rating  
16081  September 26, 2020             Idaho       2  
16082  September 26, 2020             Idaho       2  


In [4]:
df = df.drop_duplicates().reset_index(drop=True)

print("Rows after removing duplicates:", len(df))

Rows after removing duplicates: 22343


STEP 2: Convert Review_Date to datetime


In [5]:

df["Review_Date"] = pd.to_datetime(
    df["Review_Date"],
    errors="coerce"
)

print(df["Review_Date"].head())

0   2019-02-17
1   2019-02-17
2   2019-02-17
3   2019-02-16
4   2019-02-16
Name: Review_Date, dtype: datetime64[us]


STEP 3A: Check review distribution by year


In [6]:
print(
    df["Review_Date"]
    .dt.year
    .value_counts()
    .sort_index()
)

Review_Date
2018       8
2019    4717
2020    4022
2021    2043
2022    3557
2023    3754
2024    4000
2025     233
2026       9
Name: count, dtype: int64


STEP 3B: Keep only reviews from 2019 to 2024


In [7]:

df = df[
    (df["Review_Date"].dt.year >= 2019) &
    (df["Review_Date"].dt.year <= 2024)
].copy()

df.reset_index(drop=True, inplace=True)

print("Reviews remaining:", len(df))

Reviews remaining: 22093


In [8]:
# verification

print("Earliest date:", df["Review_Date"].min())
print("Latest date:", df["Review_Date"].max())

print("\nReviews per year:")
print(
    df["Review_Date"]
    .dt.year
    .value_counts()
    .sort_index()
)

Earliest date: 2019-01-03 00:00:00
Latest date: 2024-12-31 00:00:00

Reviews per year:
Review_Date
2019    4717
2020    4022
2021    2043
2022    3557
2023    3754
2024    4000
Name: count, dtype: int64


STEP 4: Create temporal columns year, month, month_year


In [9]:

df["Year"] = df["Review_Date"].dt.year

df["Month"] = df["Review_Date"].dt.month

df["Month_Year"] = df["Review_Date"].dt.to_period("M").astype(str)

df.head()

,City,Place,Page,Review_Text,Review_Date,Reviewer_Location,Rating,Year,Month,Month_Year
0,Agra,Taj Mahal,0,An amazing experience,2019-02-17,NaN,5,2019,2,2019-02
1,Agra,Taj Mahal,0,Awesome place,2019-02-17,"Dharmendra Pahadiya Pune, India",5,2019,2,2019-02
2,Agra,Taj Mahal,0,Definitely a wonder of the world,2019-02-17,NaN,5,2019,2,2019-02
3,Agra,Taj Mahal,0,Mesmerizing,2019-02-16,NaN,5,2019,2,2019-02
4,Agra,Taj Mahal,0,A place really called as wonder,2019-02-16,NaN,5,2019,2,2019-02


### Step 5: Rating Validation.

In [11]:

# Convert Rating to numeric
df["Rating"] = pd.to_numeric(
    df["Rating"],
    errors="coerce"
)

# Check rating distribution
print("Rating distribution:")
print(
    df["Rating"]
    .value_counts(dropna=False)
    .sort_index()
)

# Find missing or invalid ratings
invalid_ratings = df[
    df["Rating"].isna()
    |
    (~df["Rating"].between(1, 5))
]

print("\nTotal reviews:", len(df))
print("Invalid / missing ratings:", len(invalid_ratings))

# Show problematic rows
display(
    invalid_ratings[
        [
            "City",
            "Place",
            "Review_Text",
            "Review_Date",
            "Rating"
        ]
    ].head(20)
)

# Create a flag instead of immediately deleting reviews
df["Rating_Valid"] = df["Rating"].between(1, 5)

print("\nValid ratings:", df["Rating_Valid"].sum())
print("Invalid ratings:", (~df["Rating_Valid"]).sum())

Rating distribution:
Rating
1     1046
2      854
3     2176
4     4927
5    13090
Name: count, dtype: int64

Total reviews: 22093
Invalid / missing ratings: 0


,City,Place,Review_Text,Review_Date,Rating



Valid ratings: 22093
Invalid ratings: 0


### Step 6 — Missing-value analysis

#### Step 6A: Missing value analysis

In [12]:
missing_report = pd.DataFrame({
    "Missing_Count": df.isna().sum(),
    "Missing_Percent": (df.isna().mean() * 100).round(2)
})

display(missing_report)

,Missing_Count,Missing_Percent
City,0,0.0
Place,0,0.0
Page,0,0.0
Review_Text,0,0.0
Review_Date,0,0.0
Reviewer_Location,3358,15.2
Rating,0,0.0
Year,0,0.0
Month,0,0.0
Month_Year,0,0.0


#### Step 6B: Handle Reviewer Location

In [13]:
df["Reviewer_Location"] = (
    df["Reviewer_Location"]
    .fillna("Unknown")
    .astype(str)
    .str.strip()
)

df.loc[
    df["Reviewer_Location"] == "",
    "Reviewer_Location"
] = "Unknown"

print(
    df["Reviewer_Location"]
    .value_counts()
    .head(10)
)

Reviewer_Location
Unknown                        3358
London, United Kingdom          646
Dubai, United Arab Emirates     419
New York City, New York         207
1 contribution                  205
2 contributions                 197
3 contributions                 179
Miami, Florida                  152
Singapore, Singapore            147
Paris, France                   131
Name: count, dtype: int64


### Step 7 — Prepare review text correctly

#### Step 7A: Text Cleaning Functions

In [14]:
import re

def clean_for_model(text):
    if pd.isna(text):
        return ""

    text = str(text)

    text = re.sub(
        r"http\S+|www\S+",
        "",
        text
    )

    text = re.sub(
        r"<[^>]+>",
        " ",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    return text


def clean_for_keywords(text):
    if pd.isna(text):
        return ""

    text = str(text).lower()

    text = re.sub(
        r"http\S+|www\S+",
        "",
        text
    )

    text = re.sub(
        r"<[^>]+>",
        " ",
        text
    )

    text = re.sub(
        r"[^a-zA-Z\s]",
        " ",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    return text

#### Step 7B: Apply Text Cleaning

In [15]:
df["Review_Text"] = (
    df["Review_Text"]
    .fillna("")
    .astype(str)
)

df["Model_Text"] = (
    df["Review_Text"]
    .apply(clean_for_model)
)

df["Clean_Text"] = (
    df["Review_Text"]
    .apply(clean_for_keywords)
)

display(
    df[
        [
            "Review_Text",
            "Model_Text",
            "Clean_Text"
        ]
    ].head()
)

,Review_Text,Model_Text,Clean_Text
0,An amazing experience,An amazing experience,an amazing experience
1,Awesome place,Awesome place,awesome place
2,Definitely a wonder of the world,Definitely a wonder of the world,definitely a wonder of the world
3,Mesmerizing,Mesmerizing,mesmerizing
4,A place really called as wonder,A place really called as wonder,a place really called as wonder


### Step 8 — Remove only unusable reviews

In [16]:
empty_review_mask = (
    df["Model_Text"]
    .str.strip()
    .eq("")
)

print(
    "Empty reviews:",
    empty_review_mask.sum()
)

Empty reviews: 1


In [17]:
df = (
    df[
        ~empty_review_mask
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    "Reviews remaining:",
    len(df)
)

Reviews remaining: 22092


### Step 9 — Normalize city and attraction names

#### Step 9A: Inspect

In [18]:
print(
    sorted(
        df["City"]
        .dropna()
        .unique()
    )
)

print(
    sorted(
        df["Place"]
        .dropna()
        .unique()
    )
)

['Agra', 'Bali', 'Barcelona', 'Cairo', 'Dubai', 'Marrakech', 'Miami', 'Paris', 'Rio de Janeiro', 'Rome']
['Agra Fort', 'Arc de Triomphe', 'Bahia Palace', 'Bayside Marketplace', 'Burj Khalifa', 'Casa Batlló', 'Colosseum', 'Copacabana Beach', 'Coptic Cairo', 'Dubai Frame', 'Dubai Mall', 'Dubai Miracle Garden', 'Egyptian Museum', 'Eiffel Tower', 'Escadaria Selarón', 'Gothic Quarter', 'Ipanema Beach', 'Jardin Majorelle', 'Jemaa el-Fnaa', 'Khan el-Khalili Bazaar', 'La Rambla', 'Le Jardin Secret', 'Lempuyang Temple', 'Louvre Museum', 'Mehtab Bagh', 'Miami Beach Boardwalk', 'Miami Seaquarium', 'Montmartre', 'Mosque of Muhammad Ali', 'Mount Batur', 'Notre-Dame Cathedral', 'Pantheon', 'Park Güell', 'Saadian Tombs', 'Sacred Monkey Forest Sanctuary', 'Sagrada Familia', 'Santuário Cristo Redentor', 'Sugarloaf Mountain', 'Taj Mahal', 'Tegallalang Rice Terrace', 'The Dubai Fountain', 'The Pyramids Of Giza', 'Trevi Fountain', 'Uluwatu Temple', 'Vatican Museums', 'Vizcaya Museum & Gardens', 'Wildlife 

#### Step 9B: Basic Cleanup

In [19]:
print(
    sorted(
        df["City"]
        .dropna()
        .unique()
    )
)

print(
    sorted(
        df["Place"]
        .dropna()
        .unique()
    )
)

['Agra', 'Bali', 'Barcelona', 'Cairo', 'Dubai', 'Marrakech', 'Miami', 'Paris', 'Rio de Janeiro', 'Rome']
['Agra Fort', 'Arc de Triomphe', 'Bahia Palace', 'Bayside Marketplace', 'Burj Khalifa', 'Casa Batlló', 'Colosseum', 'Copacabana Beach', 'Coptic Cairo', 'Dubai Frame', 'Dubai Mall', 'Dubai Miracle Garden', 'Egyptian Museum', 'Eiffel Tower', 'Escadaria Selarón', 'Gothic Quarter', 'Ipanema Beach', 'Jardin Majorelle', 'Jemaa el-Fnaa', 'Khan el-Khalili Bazaar', 'La Rambla', 'Le Jardin Secret', 'Lempuyang Temple', 'Louvre Museum', 'Mehtab Bagh', 'Miami Beach Boardwalk', 'Miami Seaquarium', 'Montmartre', 'Mosque of Muhammad Ali', 'Mount Batur', 'Notre-Dame Cathedral', 'Pantheon', 'Park Güell', 'Saadian Tombs', 'Sacred Monkey Forest Sanctuary', 'Sagrada Familia', 'Santuário Cristo Redentor', 'Sugarloaf Mountain', 'Taj Mahal', 'Tegallalang Rice Terrace', 'The Dubai Fountain', 'The Pyramids Of Giza', 'Trevi Fountain', 'Uluwatu Temple', 'Vatican Museums', 'Vizcaya Museum & Gardens', 'Wildlife 

In [20]:
place_mapping = {
    "Park Guell": "Park Güell",
    "Park_Guell": "Park Güell"
}

df["Place"] = df["Place"].replace(place_mapping)

### Step 10 — Language detection

In [21]:
%pip install langdetect -q
from langdetect import detect, DetectorFactory

DetectorFactory.seed = 42

def detect_language(text):
    if not isinstance(text, str):
        return "unknown"

    text = text.strip()

    if len(text) < 20:
        return "unknown"

    try:
        return detect(text)

    except:
        return "unknown"

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [22]:
df["Language"] = (
    df["Model_Text"]
    .apply(detect_language)
)

print(
    df["Language"]
    .value_counts()
)

Language
en         11296
unknown     9281
fr           327
ro           307
de           113
af           109
it            83
nl            73
es            73
no            72
ca            71
id            64
da            42
tl            32
pt            31
et            25
cy            15
fi            14
sv            13
so             9
pl             9
hr             6
sw             6
hu             6
vi             4
tr             4
sq             2
lt             2
sk             1
cs             1
sl             1
Name: count, dtype: int64


In [23]:
df["Is_English"] = (
    df["Language"] == "en"
)

english_percentage = (
    df["Is_English"].mean()
    * 100
)

print(
    f"English reviews: {english_percentage:.2f}%"
)

English reviews: 51.13%


#### Step 10B: Language Distribution

In [24]:
city_language = pd.crosstab(
    df["City"],
    df["Language"]
)

display(city_language)

Language,af,ca,cs,cy,da,de,en,es,et,fi,...,sk,sl,so,sq,sv,sw,tl,tr,unknown,vi
City,,,,,,,,,,,,,,,,,,,,,
Agra,7,6,0,0,2,12,815,3,2,1,...,0,0,1,1,0,1,3,0,433,2
Bali,20,3,1,2,1,9,1031,9,1,3,...,0,0,1,0,1,1,2,0,911,0
Barcelona,16,10,0,3,4,9,1635,16,4,1,...,0,0,2,0,1,0,4,0,917,0
Cairo,2,10,0,0,5,5,843,3,3,2,...,0,0,0,0,0,0,3,1,815,0
Dubai,15,14,0,0,5,14,1595,7,2,0,...,0,0,2,1,1,2,7,0,1176,1
Marrakech,9,10,0,1,7,18,1350,5,4,2,...,0,0,1,0,3,0,5,0,874,0
Miami,13,7,0,4,8,11,1228,6,2,1,...,1,0,2,0,3,2,3,3,819,0
Paris,12,5,0,2,7,9,1017,5,2,2,...,0,1,0,0,4,0,1,0,1209,0
Rio de Janeiro,6,2,0,3,2,10,723,12,1,1,...,0,0,0,0,0,0,3,0,917,0


In [25]:
city_language_pct = pd.crosstab(
    df["City"],
    df["Language"],
    normalize="index"
) * 100

display(city_language_pct.round(2))

Language,af,ca,cs,cy,da,de,en,es,et,fi,...,sk,sl,so,sq,sv,sw,tl,tr,unknown,vi
City,,,,,,,,,,,,,,,,,,,,,
Agra,0.52,0.45,0.00,0.00,0.15,0.89,60.73,0.22,0.15,0.07,...,0.00,0.00,0.07,0.07,0.00,0.07,0.22,0.00,32.27,0.15
Bali,0.94,0.14,0.05,0.09,0.05,0.42,48.31,0.42,0.05,0.14,...,0.00,0.00,0.05,0.00,0.05,0.05,0.09,0.00,42.69,0.00
Barcelona,0.59,0.37,0.00,0.11,0.15,0.33,60.00,0.59,0.15,0.04,...,0.00,0.00,0.07,0.00,0.04,0.00,0.15,0.00,33.65,0.00
Cairo,0.11,0.56,0.00,0.00,0.28,0.28,47.63,0.17,0.17,0.11,...,0.00,0.00,0.00,0.00,0.00,0.00,0.17,0.06,46.05,0.00
Dubai,0.51,0.47,0.00,0.00,0.17,0.47,54.09,0.24,0.07,0.00,...,0.00,0.00,0.07,0.03,0.03,0.07,0.24,0.00,39.88,0.03
Marrakech,0.37,0.41,0.00,0.04,0.29,0.74,55.37,0.21,0.16,0.08,...,0.00,0.00,0.04,0.00,0.12,0.00,0.21,0.00,35.85,0.00
Miami,0.59,0.32,0.00,0.18,0.36,0.50,55.79,0.27,0.09,0.05,...,0.05,0.00,0.09,0.00,0.14,0.09,0.14,0.14,37.21,0.00
Paris,0.51,0.21,0.00,0.08,0.30,0.38,42.91,0.21,0.08,0.08,...,0.00,0.04,0.00,0.00,0.17,0.00,0.04,0.00,51.01,0.00
Rio de Janeiro,0.34,0.11,0.00,0.17,0.11,0.57,41.24,0.68,0.06,0.06,...,0.00,0.00,0.00,0.00,0.00,0.00,0.17,0.00,52.31,0.00


#### Step10D: Investigate "unknown" reviews

In [30]:
# Unknown Reviews By City

unknown_by_city = (
    df[df["Language"] == "unknown"]
    .groupby("City")
    .size()
    .reset_index(name="Unknown_Count")
)

total_by_city = (
    df.groupby("City")
    .size()
    .reset_index(name="Total_Reviews")
)

unknown_summary = total_by_city.merge(
    unknown_by_city,
    on="City",
    how="left"
)

unknown_summary["Unknown_Count"] = (
    unknown_summary["Unknown_Count"].fillna(0).astype(int)
)

unknown_summary["Unknown_Percentage"] = (
    unknown_summary["Unknown_Count"]
    / unknown_summary["Total_Reviews"]
    * 100
).round(2)

display(unknown_summary)

,City,Total_Reviews,Unknown_Count,Unknown_Percentage
0,Agra,1342,433,32.27
1,Bali,2134,911,42.69
2,Barcelona,2725,917,33.65
3,Cairo,1770,815,46.05
4,Dubai,2949,1176,39.88
5,Marrakech,2438,874,35.85
6,Miami,2201,819,37.21
7,Paris,2370,1209,51.01
8,Rio de Janeiro,1753,917,52.31
9,Rome,2410,1210,50.21


In [31]:
unknown_examples = (
    df[df["Language"] == "unknown"]
    .groupby("City", group_keys=False)
    .head(10)
)

display(
    unknown_examples[
        ["City", "Place", "Review_Text", "Language"]
    ]
)

,City,Place,Review_Text,Language
1,Agra,Taj Mahal,Awesome place,unknown
3,Agra,Taj Mahal,Mesmerizing,unknown
6,Agra,Taj Mahal,Jaw dropping wonder,unknown
11,Agra,Taj Mahal,Amazing place,unknown
14,Agra,Taj Mahal,Its got to be done!,unknown
...,...,...,...,...
19694,Rome,Trevi Fountain,Beautiful,unknown
19695,Rome,Trevi Fountain,Trevi,unknown
19696,Rome,Trevi Fountain,April,unknown
19697,Rome,Trevi Fountain,Too many tourists!,unknown


In [ ]:
# Unknown Reviews By City
unknown_by_city = (
    df[df["Language"] == "unknown"]
    .groupby("City")
    .size()
    .reset_index(name="Unknown_Count")
)

total_by_city = (
    df.groupby("City")
    .size()
    .reset_index(name="Total_Reviews")
)

unknown_summary = total_by_city.merge(
    unknown_by_city,
    on="City",
    how="left"
)

unknown_summary["Unknown_Count"] = (
    unknown_summary["Unknown_Count"].fillna(0).astype(int)
)

unknown_summary["Unknown_Percentage"] = (
    unknown_summary["Unknown_Count"]
    / unknown_summary["Total_Reviews"]
    * 100
).round(2)

display(unknown_summary)

,City,Total_Reviews,Unknown_Count,Unknown_Percentage
0,Agra,1342,433,32.27
1,Bali,2134,911,42.69
2,Barcelona,2725,917,33.65
3,Cairo,1770,815,46.05
4,Dubai,2949,1176,39.88
5,Marrakech,2438,874,35.85
6,Miami,2201,819,37.21
7,Paris,2370,1209,51.01
8,Rio de Janeiro,1753,917,52.31
9,Rome,2410,1210,50.21


In [ ]:
# Check Total Unknown Language Reviews

# Total number of reviews
total_reviews = len(df)

# Number of reviews detected as "unknown"
unknown_reviews = (df["Language"] == "unknown").sum()

# Percentage of unknown reviews
unknown_percentage = (unknown_reviews / total_reviews) * 100

print(f"Total reviews: {total_reviews:,}")
print(f"Unknown reviews: {unknown_reviews:,}")
print(f"Unknown percentage: {unknown_percentage:.2f}%")

print(
    f"\n{unknown_reviews:,} of {total_reviews:,} reviews "
    f"({unknown_percentage:.2f}%) are unknown."
)

Total reviews: 22,092
Unknown reviews: 9,281
Unknown percentage: 42.01%

9,281 of 22,092 reviews (42.01%) are unknown.


In [ ]:
 Analyse Unknown Reviews By Text Length

df["Word_Count"] = (
    df["Review_Text"]
    .fillna("")
    .astype(str)
    .str.split()
    .str.len()
)

unknown_df = df[df["Language"] == "unknown"]

print("Total unknown:", len(unknown_df))

print("\nUnknown reviews by word length:")
print("1-2 words :", (unknown_df["Word_Count"] <= 2).sum())
print("3-5 words :", unknown_df["Word_Count"].between(3, 5).sum())
print("6-10 words:", unknown_df["Word_Count"].between(6, 10).sum())
print("11-20 words:", unknown_df["Word_Count"].between(11, 20).sum())
print(">20 words :", (unknown_df["Word_Count"] > 20).sum())

Total unknown: 9281

Unknown reviews by word length:
1-2 words : 5892
3-5 words : 3386
6-10 words: 3
11-20 words: 0
>20 words : 0


### Step 10E: Create Language Status

In [ ]:

def create_language_status(row):
    
    if row["Language"] != "unknown":
        return "detected"
    
    if row["Word_Count"] <= 5:
        return "unknown_short"
    
    return "unknown_other"


df["Language_Status"] = df.apply(
    create_language_status,
    axis=1
)

print(df["Language_Status"].value_counts())

print("\nPercentage:")
print(
    (df["Language_Status"].value_counts(normalize=True) * 100)
    .round(2)
)

Language_Status
detected         12811
unknown_short     9278
unknown_other        3
Name: count, dtype: int64

Percentage:
Language_Status
detected         57.99
unknown_short    42.00
unknown_other     0.01
Name: proportion, dtype: float64


In [ ]:
# Check unknown_other reviews
display(
    df[df["Language_Status"] == "unknown_other"][
        [
            "City",
            "Place",
            "Review_Text",
            "Word_Count",
            "Language"
        ]
    ]
)

,City,Place,Review_Text,Word_Count,Language
9038,Marrakech,Jemaa el-Fnaa,Between a 3 and a 4,6,unknown
17957,Rio de Janeiro,Sugarloaf Mountain,A must do in Rio !,6,unknown
18177,Rio de Janeiro,Sugarloaf Mountain,A must to do in Rio,6,unknown


### Step 10F: Final Language Quality Summary

In [ ]:
total = len(df)

summary = pd.DataFrame({
    "Count": df["Language_Status"].value_counts(),
    "Percentage": (
        df["Language_Status"].value_counts(normalize=True) * 100
    ).round(2)
})

print("FINAL LANGUAGE STATUS")
display(summary)

print("\nDetected languages:")
print(
    df.loc[df["Language_Status"] == "detected", "Language"]
      .value_counts()
      .head(15)
)

print("\nDataset size:", f"{total:,}")

FINAL LANGUAGE STATUS


,Count,Percentage
Language_Status,,
detected,12811,57.99
unknown_short,9278,42.00
unknown_other,3,0.01



Detected languages:
Language
en    11296
fr      327
ro      307
de      113
af      109
it       83
nl       73
es       73
no       72
ca       71
id       64
da       42
tl       32
pt       31
et       25
Name: count, dtype: int64

Dataset size: 22,092


In [38]:
# ============================================================
# STEP 11: SHORT REVIEW QUALITY ANALYSIS
# ============================================================

df["Very_Short_Review"] = df["Word_Count"] <= 5

print("Very short reviews:")
print(df["Very_Short_Review"].value_counts())

print("\nPercentage:")
print(
    (df["Very_Short_Review"].value_counts(normalize=True) * 100)
    .round(2)
)

# Short reviews by city
short_by_city = (
    df.groupby("City")["Very_Short_Review"]
      .agg(["sum", "count", "mean"])
)

short_by_city["Percentage"] = (
    short_by_city["mean"] * 100
).round(2)

display(short_by_city[["sum", "count", "Percentage"]])

Very short reviews:
Very_Short_Review
True     15112
False     6980
Name: count, dtype: int64

Percentage:
Very_Short_Review
True     68.4
False    31.6
Name: proportion, dtype: float64


,sum,count,Percentage
City,,,
Agra,796,1342,59.31
Bali,1535,2134,71.93
Barcelona,1727,2725,63.38
Cairo,1280,1770,72.32
Dubai,1853,2949,62.83
Marrakech,1557,2438,63.86
Miami,1405,2201,63.83
Paris,1774,2370,74.85
Rio de Janeiro,1348,1753,76.90


In [39]:
short_by_year = (
    df.groupby("Year")["Very_Short_Review"]
      .agg(["sum", "count", "mean"])
)

short_by_year["Percentage"] = (
    short_by_year["mean"] * 100
).round(2)

display(short_by_year[["sum", "count", "Percentage"]])

,sum,count,Percentage
Year,,,
2019,3325,4717,70.49
2020,2818,4022,70.06
2021,1398,2043,68.43
2022,2363,3557,66.43
2023,2510,3754,66.86
2024,2698,3999,67.47


In [40]:
df["Very_Short_Review"] = df["Word_Count"] <= 5

### Step 11 — Handle short reviews correctly

In [41]:
print(df["Very_Short_Review"].value_counts())

print("\nPercentage:")
print(
    df["Very_Short_Review"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Very_Short_Review
True     15112
False     6980
Name: count, dtype: int64

Percentage:
Very_Short_Review
True     68.4
False    31.6
Name: proportion, dtype: float64


In [42]:
df = df[
    df["Review_Text"]
    .fillna("")
    .astype(str)
    .str.strip()
    .ne("")
].copy()

df.reset_index(drop=True, inplace=True)

print("Reviews remaining:", len(df))

Reviews remaining: 22092


Step 12 — Add a unique Review ID

In [43]:
df = df.reset_index(drop=True)

df["Review_ID"] = [
    f"R{i:06d}"
    for i in range(1, len(df) + 1)
]

df[
    [
        "Review_ID",
        "City",
        "Place",
        "Review_Date",
        "Review_Text"
    ]
].head(10)

,Review_ID,City,Place,Review_Date,Review_Text
0,R000001,Agra,Taj Mahal,2019-02-17,An amazing experience
1,R000002,Agra,Taj Mahal,2019-02-17,Awesome place
2,R000003,Agra,Taj Mahal,2019-02-17,Definitely a wonder of the world
3,R000004,Agra,Taj Mahal,2019-02-16,Mesmerizing
4,R000005,Agra,Taj Mahal,2019-02-16,A place really called as wonder
5,R000006,Agra,Taj Mahal,2019-02-16,It's was a magnificent experience to be seeing...
6,R000007,Agra,Taj Mahal,2019-02-16,Jaw dropping wonder
7,R000008,Agra,Taj Mahal,2019-02-16,No wonder it's a wonder!
8,R000009,Agra,Taj Mahal,2019-02-16,An ambition acheived
9,R000010,Agra,Taj Mahal,2019-02-16,This is definitely one of the most amazing pla...


#### Step 12A: Examine Complete Language Distribution

In [45]:
language_summary = (
    df["Language"]
    .value_counts(dropna=False)
    .rename_axis("Language")
    .reset_index(name="Review_Count")
)

language_summary["Percentage"] = (
    language_summary["Review_Count"]
    / len(df) * 100
).round(2)

language_summary

,Language,Review_Count,Percentage
0,en,11296,51.13
1,unknown,9281,42.01
2,fr,327,1.48
3,ro,307,1.39
4,de,113,0.51
5,af,109,0.49
6,it,83,0.38
7,nl,73,0.33
8,es,73,0.33
9,no,72,0.33


#### Step 12B — Separate reliable language from uncertain short text

In [46]:
df["Language_Status"] = "detected"

df.loc[
    (df["Language"] == "unknown") &
    (df["Word_Count"] <= 5),
    "Language_Status"
] = "short_text_uncertain"

df.loc[
    (df["Language"] == "unknown") &
    (df["Word_Count"] > 5),
    "Language_Status"
] = "unknown_longer_text"

print(df["Language_Status"].value_counts())

print("\nPercentages:")
print(
    df["Language_Status"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Language_Status
detected                12811
short_text_uncertain     9278
unknown_longer_text         3
Name: count, dtype: int64

Percentages:
Language_Status
detected                57.99
short_text_uncertain    42.00
unknown_longer_text      0.01
Name: proportion, dtype: float64


#### Step 12C — Inspect the non-English reviews

In [47]:
non_english = df[
    (~df["Language"].isin(["en", "unknown"]))
]

print("Detected non-English reviews:", len(non_english))

non_english[
    [
        "Review_ID",
        "City",
        "Place",
        "Language",
        "Word_Count",
        "Review_Text"
    ]
].sample(min(30, len(non_english)), random_state=42)

Detected non-English reviews: 1515


,Review_ID,City,Place,Language,Word_Count,Review_Text
774,R000775,Agra,Agra Fort,ca,7,Explore Agra: Agra Fort – Khas Mahal
2787,R002788,Barcelona,La Rambla,ro,4,Nicest place in Barcelona
21350,R021351,Rome,Pantheon,af,5,Outdoor Museum of Roman Ruins
12989,R012990,Bali,Uluwatu Temple,fi,5,"Uluwatu monkeys a nuisance, avoid"
6667,R006668,Dubai,Dubai Frame,ro,3,Beautiful building ??
15229,R015230,Cairo,Coptic Cairo,af,3,very interesting visit
8549,R008550,Marrakech,Le Jardin Secret,fr,3,Beautiful quiet garden
3443,R003444,Barcelona,Casa Batlló,ro,3,Fantastic Gaudi Architecture
21348,R021349,Rome,Pantheon,ro,3,Beautiful Historical Forum
20619,R020620,Rome,Colosseum,de,5,Colosseum underground and Roman Forum


In [48]:
# STEP 12D — Language distribution for longer reviews

long_reviews = df[df["Word_Count"] >= 10].copy()

print("Total dataset:", len(df))
print("Reviews with >=10 words:", len(long_reviews))

print(
    "Percentage with >=10 words:",
    round(len(long_reviews) / len(df) * 100, 2),
    "%"
)

print("\nLanguage distribution for reviews >=10 words:")

long_language_summary = (
    long_reviews["Language"]
    .value_counts()
    .rename_axis("Language")
    .reset_index(name="Count")
)

long_language_summary["Percentage"] = (
    long_language_summary["Count"]
    / len(long_reviews) * 100
).round(2)

display(long_language_summary.head(20))

Total dataset: 22092
Reviews with >=10 words: 2760
Percentage with >=10 words: 12.49 %

Language distribution for reviews >=10 words:


,Language,Count,Percentage
0,en,2745,99.46
1,af,7,0.25
2,fr,3,0.11
3,de,2,0.07
4,sv,1,0.04
5,cy,1,0.04
6,pt,1,0.04


In [49]:
# Inspect non-English reviews where there is enough text
long_non_english = df[
    (df["Word_Count"] >= 10) &
    (~df["Language"].isin(["en", "unknown"]))
].copy()

print(
    "Non-English detected reviews with >=10 words:",
    len(long_non_english)
)

display(
    long_non_english[
        [
            "Review_ID",
            "City",
            "Place",
            "Language",
            "Word_Count",
            "Review_Text"
        ]
    ].sample(
        min(30, len(long_non_english)),
        random_state=42
    )
)

Non-English detected reviews with >=10 words: 15


,Review_ID,City,Place,Language,Word_Count,Review_Text
9842,R009843,Miami,Wynwood Walls,cy,12,You e not done Miami if you e not done Wynwood...
18344,R018345,Rio de Janeiro,Sugarloaf Mountain,af,11,Indeed Beautiful over looking views on top of ...
811,R000812,Agra,Agra Fort,de,10,RED FORT - FORTRESS WITH VIEW OF THE TAJ MAHAL
19469,R019470,Rio de Janeiro,Santuário Cristo Redentor,pt,12,A must to see if you go to Rio de Janeiro !
6715,R006716,Dubai,Dubai Frame,af,13,"Must visit in Dubai! ????? ?????????? ?????, ?..."
9451,R009452,Marrakech,Jemaa el-Fnaa,sv,10,Fresh juice stalls in jemaa el fna (avoid stal...
1961,R001962,Barcelona,Park Güell,af,10,A must-visit. Feels like transported into Hans...
868,R000869,Agra,Agra Fort,af,10,MUST visit in Agra. Make sure to get a guide
20446,R020447,Rome,Colosseum,fr,10,Do not use Tiqets or Omnia if you visit Colosseum
6705,R006706,Dubai,Dubai Frame,de,10,"ONE OF DUBAI'S NEWEST ATTRACTIONS, AMAZING VIE..."


### STEP 13 — Final Dara Quality Audit

In [51]:


print("=" * 60)
print("FINAL DATA QUALITY AUDIT")
print("=" * 60)

# Dataset size
print("\n1. DATASET SIZE")
print("Rows:", len(df))
print("Columns:", len(df.columns))


# Duplicate check
print("\n2. DUPLICATES")
print("Exact duplicate rows:", df.duplicated().sum())


# Review ID check
print("\n3. REVIEW ID")
print("Unique Review_ID:", df["Review_ID"].nunique())
print("Duplicate Review_ID:", df["Review_ID"].duplicated().sum())


# Date check
print("\n4. DATE RANGE")
print("Earliest:", df["Review_Date"].min())
print("Latest:", df["Review_Date"].max())


# Rating check
print("\n5. RATINGS")
print(df["Rating"].value_counts().sort_index())

print(
    "Invalid ratings:",
    (~df["Rating"].between(1, 5)).sum()
)


# Missing values
print("\n6. MISSING VALUES")

missing = pd.DataFrame({
    "Missing_Count": df.isna().sum(),
    "Missing_Percentage":
        (df.isna().mean() * 100).round(2)
})

print(
    missing[
        missing["Missing_Count"] > 0
    ]
)


# Cities
print("\n7. CITIES")
print("Number of cities:", df["City"].nunique())

print(
    df["City"]
    .value_counts()
    .sort_index()
)


# Places
print("\n8. ATTRACTIONS")
print("Number of attractions:", df["Place"].nunique())


# Years
print("\n9. REVIEWS PER YEAR")

print(
    df["Year"]
    .value_counts()
    .sort_index()
)


# Short reviews
print("\n10. SHORT REVIEWS")

print(
    "Reviews <=5 words:",
    df["Very_Short_Review"].sum()
)

print(
    "Percentage:",
    round(
        df["Very_Short_Review"].mean() * 100,
        2
    ),
    "%"
)




FINAL DATA QUALITY AUDIT

1. DATASET SIZE
Rows: 22092
Columns: 19

2. DUPLICATES
Exact duplicate rows: 0

3. REVIEW ID
Unique Review_ID: 22092
Duplicate Review_ID: 0

4. DATE RANGE
Earliest: 2019-01-03 00:00:00
Latest: 2024-12-31 00:00:00

5. RATINGS
Rating
1     1046
2      854
3     2176
4     4927
5    13089
Name: count, dtype: int64
Invalid ratings: 0

6. MISSING VALUES
Empty DataFrame
Columns: [Missing_Count, Missing_Percentage]
Index: []

7. CITIES
Number of cities: 10
City
Agra              1342
Bali              2134
Barcelona         2725
Cairo             1770
Dubai             2949
Marrakech         2438
Miami             2201
Paris             2370
Rio de Janeiro    1753
Rome              2410
Name: count, dtype: int64

8. ATTRACTIONS
Number of attractions: 48

9. REVIEWS PER YEAR
Year
2019    4717
2020    4022
2021    2043
2022    3557
2023    3754
2024    3999
Name: count, dtype: int64

10. SHORT REVIEWS
Reviews <=5 words: 15112
Percentage: 68.4 %


#### Step 14: 

#### Step 14A:  City × year coverage

In [52]:
city_year = pd.crosstab(
    df["City"],
    df["Year"]
)

display(city_year)

Year,2019,2020,2021,2022,2023,2024
City,,,,,,
Agra,435,251,85,179,185,207
Bali,734,323,42,320,351,364
Barcelona,500,587,154,580,394,510
Cairo,424,252,218,233,284,359
Dubai,496,482,471,490,531,479
Marrakech,500,375,101,429,482,551
Miami,577,385,310,304,333,292
Paris,312,601,221,424,386,426
Rio de Janeiro,382,322,151,174,357,367


#### City × month coverage

In [53]:
monthly_counts = (
    df.groupby(
        ["City", "Year", "Month"]
    )
    .size()
    .reset_index(name="Review_Count")
)

display(monthly_counts.head(20))

print("Number of observed city-months:",
      len(monthly_counts))

print("Minimum reviews/month:",
      monthly_counts["Review_Count"].min())

print("Median reviews/month:",
      monthly_counts["Review_Count"].median())

print("Mean reviews/month:",
      round(monthly_counts["Review_Count"].mean(), 2))

print("Maximum reviews/month:",
      monthly_counts["Review_Count"].max())

,City,Year,Month,Review_Count
0,Agra,2019,1,10
1,Agra,2019,2,146
2,Agra,2019,3,67
3,Agra,2019,4,22
4,Agra,2019,5,38
5,Agra,2019,6,41
6,Agra,2019,7,26
7,Agra,2019,8,22
8,Agra,2019,9,20
9,Agra,2019,10,19


Number of observed city-months: 675
Minimum reviews/month: 1
Median reviews/month: 25.0
Mean reviews/month: 32.73
Maximum reviews/month: 213


#### STEP 14B — Find Missing City _ Month

In [54]:
all_months = pd.MultiIndex.from_product(
    [
        sorted(df["City"].unique()),
        range(2019, 2025),
        range(1, 13)
    ],
    names=["City", "Year", "Month"]
)

monthly_complete = (
    monthly_counts
    .set_index(["City", "Year", "Month"])
    .reindex(all_months, fill_value=0)
    .reset_index()
)

missing_months = monthly_complete[
    monthly_complete["Review_Count"] == 0
].copy()

print("Expected city-months:", len(monthly_complete))
print(
    "Observed city-months:",
    (monthly_complete["Review_Count"] > 0).sum()
)
print("Missing city-months:", len(missing_months))

print(
    "Coverage:",
    round(
        (monthly_complete["Review_Count"] > 0).mean() * 100,
        2
    ),
    "%"
)

display(missing_months)

Expected city-months: 720
Observed city-months: 675
Missing city-months: 45
Coverage: 93.75 %


,City,Year,Month,Review_Count
17,Agra,2020,6,0
23,Agra,2020,12,0
29,Agra,2021,6,0
72,Bali,2019,1,0
102,Bali,2021,7,0
152,Barcelona,2019,9,0
153,Barcelona,2019,10,0
154,Barcelona,2019,11,0
155,Barcelona,2019,12,0
170,Barcelona,2021,3,0


In [ ]:
# check missing months by city
missing_by_city = (
    missing_months
    .groupby("City")
    .size()
    .reset_index(name="Missing_Months")
    .sort_values("Missing_Months", ascending=False)
)

display(missing_by_city)

,City,Missing_Months
6,Paris,9
8,Rome,8
5,Marrakech,7
2,Barcelona,6
4,Dubai,5
0,Agra,3
3,Cairo,3
1,Bali,2
7,Rio de Janeiro,2


Step 14C — Investigate low-sample months

In [56]:

thresholds = [1, 5, 10, 20, 30]

for threshold in thresholds:

    count = (
        monthly_complete["Review_Count"] < threshold
    ).sum()

    percentage = (
        count / len(monthly_complete) * 100
    )

    print(
        f"< {threshold:2d} reviews : "
        f"{count:3d} city-months "
        f"({percentage:.2f}%)"
    )

<  1 reviews :  45 city-months (6.25%)
<  5 reviews :  93 city-months (12.92%)
< 10 reviews : 169 city-months (23.47%)
< 20 reviews : 312 city-months (43.33%)
< 30 reviews : 448 city-months (62.22%)


In [57]:
observed_months = monthly_complete[
    monthly_complete["Review_Count"] > 0
].copy()

print("ONLY MONTHS THAT CONTAIN REVIEWS\n")

for threshold in [5, 10, 20, 30]:

    count = (
        observed_months["Review_Count"] < threshold
    ).sum()

    percentage = (
        count / len(observed_months) * 100
    )

    print(
        f"1-{threshold-1} reviews: "
        f"{count} months "
        f"({percentage:.2f}%)"
    )

ONLY MONTHS THAT CONTAIN REVIEWS

1-4 reviews: 48 months (7.11%)
1-9 reviews: 124 months (18.37%)
1-19 reviews: 267 months (39.56%)
1-29 reviews: 403 months (59.70%)


### STEP 14D — MONTHLY COVERAGE BY YEAR

In [58]:

year_quality = (
    monthly_complete
    .groupby("Year")
    .agg(
        Total_City_Months=("Review_Count", "size"),
        Months_With_Data=("Review_Count",
                          lambda x: (x > 0).sum()),
        Missing_Months=("Review_Count",
                        lambda x: (x == 0).sum()),
        Low_Under_10=("Review_Count",
                     lambda x: ((x > 0) & (x < 10)).sum()),
        Median_Reviews=("Review_Count", "median"),
        Total_Reviews=("Review_Count", "sum")
    )
)

display(year_quality)

,Total_City_Months,Months_With_Data,Missing_Months,Low_Under_10,Median_Reviews,Total_Reviews
Year,,,,,,
2019,120,86,34,9,20.5,4717
2020,120,116,4,39,15.0,4022
2021,120,115,5,41,13.0,2043
2022,120,120,0,20,25.0,3557
2023,120,119,1,9,28.0,3754
2024,120,119,1,6,29.0,3999


In [59]:
missing_2019 = monthly_complete[
    (monthly_complete["Year"] == 2019) &
    (monthly_complete["Review_Count"] == 0)
]

display(
    missing_2019[
        ["City", "Year", "Month", "Review_Count"]
    ].sort_values(["City", "Month"])
)

,City,Year,Month,Review_Count
72,Bali,2019,1,0
152,Barcelona,2019,9,0
153,Barcelona,2019,10,0
154,Barcelona,2019,11,0
155,Barcelona,2019,12,0
218,Cairo,2019,3,0
224,Cairo,2019,9,0
288,Dubai,2019,1,0
296,Dubai,2019,9,0
297,Dubai,2019,10,0


In [60]:
missing_2019_summary = (
    missing_2019
    .groupby("City")
    .size()
    .reset_index(name="Missing_Months_2019")
    .sort_values("Missing_Months_2019", ascending=False)
)

display(missing_2019_summary)

,City,Missing_Months_2019
5,Paris,9
7,Rome,7
4,Marrakech,5
1,Barcelona,4
3,Dubai,4
2,Cairo,2
6,Rio de Janeiro,2
0,Bali,1
